# Patch (Update) a Local Unit

**Environment:** Staging (`goadmin-stage.ifrc.org`)

This notebook updates **specific fields** of an existing local unit using a `PATCH` request.
Unlike `PUT`, a `PATCH` only changes the fields you include in the request body.

> **This notebook targets the STAGING environment to protect production data.**

---

## 1. Setup & Authentication

In [1]:
import requests
import json
import getpass

# ============================================================
# ENVIRONMENT - This notebook uses STAGING (safe for testing)
# ============================================================
BASE_URL = "https://goadmin-stage.ifrc.org/api/v2"

# Securely input your API token
TOKEN = getpass.getpass("Enter your IFRC GO STAGING API token: ")

HEADERS = {
    "Authorization": f"Token {TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json"
}

print("✅ Authentication configured")
print("🟠 Environment: STAGING (goadmin-stage.ifrc.org)")
print("   Production data will NOT be affected.")

✅ Authentication configured
🟠 Environment: STAGING (goadmin-stage.ifrc.org)
   Production data will NOT be affected.


## 2. Configuration

Set the unit ID you want to update, and the fields to change.

In [8]:
# ============================================================
# CONFIGURATION - Change these values
# ============================================================

UNIT_ID = 117111   # ID of the local unit to update on STAGING

# Fields to update via PATCH (only include fields you want to change)
# Common updatable fields:
#   - local_branch_name:       Branch name in local language
#   - english_branch_name:     Branch name in English
#   - address_loc:             Address in local language
#   - address_en:              Address in English
#   - city_loc:                City in local language
#   - city_en:                 City in English
#   - phone:                   Contact phone number
#   - email:                   Contact email
#   - update_reason_overview:  Reason for the update

UPDATE_FIELDS = {
    "local_branch_name": "Beispiel Ortsverein",
    "english_branch_name": "Example Local Branch",
    "update_reason_overview": "Testing PATCH update from notebook."
}

print(f"🎯 Target Unit ID: {UNIT_ID}")
print(f"📝 Fields to update:")
for key, value in UPDATE_FIELDS.items():
    print(f"   • {key}: {value}")

🎯 Target Unit ID: 117111
📝 Fields to update:
   • local_branch_name: Beispiel Ortsverein
   • english_branch_name: Example Local Branch
   • update_reason_overview: Testing PATCH update from notebook.


## 3. Fetch Current Unit Data

First, let's see the current state of the unit before making changes.

In [9]:
url = f"{BASE_URL}/local-units/{UNIT_ID}/"

print(f"🔄 Fetching current data for unit {UNIT_ID}...")
print(f"   URL: {url}\n")

try:
    get_response = requests.get(url, headers=HEADERS)
    get_response.raise_for_status()
    current_data = get_response.json()
    
    print(f"✅ Current unit data:")
    print(f"   ID:                  {current_data.get('id')}")
    print(f"   local_branch_name:   {current_data.get('local_branch_name')}")
    print(f"   english_branch_name: {current_data.get('english_branch_name')}")
    print(f"   Type:                {current_data.get('type')} ({current_data.get('type_details', {}).get('name', 'N/A')})")
    print(f"   Status:              {current_data.get('status')} ({current_data.get('status_details', 'N/A')})")
    print(f"   Country:             {current_data.get('country')} ({current_data.get('country_details', {}).get('name', 'N/A')})")
    print(f"   Level:               {current_data.get('level')}")
    print(f"   Address (local):     {current_data.get('address_loc')}")
    print(f"   Address (en):        {current_data.get('address_en')}")
    
except requests.exceptions.HTTPError as e:
    print(f"❌ HTTP Error: {e}")
    print(f"   Response: {get_response.text[:500]}")
    current_data = None
except requests.exceptions.RequestException as e:
    print(f"❌ Request Error: {e}")
    current_data = None

🔄 Fetching current data for unit 117111...
   URL: https://goadmin-stage.ifrc.org/api/v2/local-units/117111/

✅ Current unit data:
   ID:                  117111
   local_branch_name:   Bezirksamt Berlin
   english_branch_name: Berlin Administartive Center
   Type:                1 (Administrative)
   Status:              1 (Validated)
   Country:             72 (Germany)
   Level:               1
   Address (local):     Buckstrasse Berlin
   Address (en):        Buckstrasse Berlin


## 4. Prepare PATCH Payload

The IFRC GO API may require certain fields (type, country, level, location_json)
to be present even in a PATCH request. We include existing values as a workaround.

In [10]:
if current_data is None:
    print("⚠️ Cannot prepare payload — current data not available.")
    print("   Please re-run the cell above to fetch unit data.")
else:
    patch_payload = UPDATE_FIELDS.copy()
    
    # Include required fields from current data (workaround for API validation)
    patch_payload["type"] = current_data.get("type")
    patch_payload["country"] = current_data.get("country")
    patch_payload["level"] = current_data.get("level")
    
    # Convert location_geojson to location_json format
    # The API returns location_geojson but expects location_json in write requests
    location_geojson = current_data.get("location_geojson")
    if location_geojson and location_geojson.get("coordinates"):
        coords = location_geojson["coordinates"]
        patch_payload["location_json"] = {
            "lng": coords[0],
            "lat": coords[1]
        }
    elif current_data.get("location_json"):
        patch_payload["location_json"] = current_data.get("location_json")
    
    print("📦 Final PATCH payload:")
    print(json.dumps(patch_payload, indent=2, ensure_ascii=False))

📦 Final PATCH payload:
{
  "local_branch_name": "Beispiel Ortsverein",
  "english_branch_name": "Example Local Branch",
  "update_reason_overview": "Testing PATCH update from notebook.",
  "type": 1,
  "country": 72,
  "level": 1,
  "location_json": {
    "lng": 13.4,
    "lat": 52.52
  }
}


## 5. Send PATCH Request

🟠 **This will modify data on STAGING.** Review the payload above before running.

In [11]:
if current_data is None:
    print("⚠️ Skipping — no current data available.")
else:
    print(f"🔄 Sending PATCH to: {url}")
    print(f"   Environment: STAGING\n")
    
    try:
        response = requests.patch(url, headers=HEADERS, json=patch_payload)
        
        print(f"📡 Response:")
        print(f"   Status Code: {response.status_code}")
        print(f"   Content-Type: {response.headers.get('Content-Type', 'N/A')}")
        
        if response.status_code in [200, 204]:
            print(f"\n✅ SUCCESS — Unit {UNIT_ID} updated on staging!")
            
            if response.status_code == 200:
                try:
                    result = response.json()
                    print(f"\n📝 Updated values:")
                    for key in UPDATE_FIELDS:
                        print(f"   • {key}: {result.get(key)}")
                except json.JSONDecodeError:
                    print(f"   (Could not parse response body)")
        else:
            print(f"\n❌ FAILED — Status {response.status_code}")
            print(f"   Response: {response.text[:500]}")
            
            try:
                error_data = response.json()
                if isinstance(error_data, dict):
                    for key, value in error_data.items():
                        print(f"   ⚠️ {key}: {value}")
            except json.JSONDecodeError:
                pass
                
    except requests.exceptions.RequestException as e:
        print(f"❌ Request Error: {e}")

🔄 Sending PATCH to: https://goadmin-stage.ifrc.org/api/v2/local-units/117111/
   Environment: STAGING

📡 Response:
   Status Code: 200
   Content-Type: application/json

✅ SUCCESS — Unit 117111 updated on staging!

📝 Updated values:
   • local_branch_name: Beispiel Ortsverein
   • english_branch_name: Example Local Branch
   • update_reason_overview: Testing PATCH update from notebook.


## 6. Verify the Update

Fetch the unit again to confirm the changes were applied.

In [12]:
print(f"🔄 Verifying update for unit {UNIT_ID}...\n")

try:
    verify_response = requests.get(url, headers=HEADERS)
    verify_response.raise_for_status()
    updated_data = verify_response.json()
    
    print(f"✅ Verification — current values on staging:")
    for key in UPDATE_FIELDS:
        old_val = current_data.get(key) if current_data else "N/A"
        new_val = updated_data.get(key)
        changed = "✓ changed" if old_val != new_val else "(unchanged)"
        print(f"   • {key}: {new_val}  {changed}")
        
except requests.exceptions.RequestException as e:
    print(f"❌ Verification failed: {e}")

🔄 Verifying update for unit 117111...

✅ Verification — current values on staging:
   • local_branch_name: Beispiel Ortsverein  ✓ changed
   • english_branch_name: Example Local Branch  ✓ changed
   • update_reason_overview: Testing PATCH update from notebook.  ✓ changed
